# 03 — Modelagem e seleção / Modeling and selection

**PT:** Este notebook compara modelos em uma divisão fixa de validação e usa validação cruzada para ajustar o Random Forest. O melhor estimador é salvo em `artifacts/`.

**EN:** This notebook compares models on a fixed validation split and uses cross-validation to tune Random Forest. The best estimator is saved in `artifacts/`.

**Pré-requisito / Prerequisite:** execute `02_preparacao_atributos.ipynb` first.

In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

DATA_DIR = Path('../DATA/processed')
ARTIFACT_DIR = Path('../artifacts')
ARTIFACT_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42

train = pd.read_csv(DATA_DIR / 'train_prepared.csv')
X = train.drop(columns=['PassengerId', 'Survived'])
y = train['Survived']
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

## Modelos de referência / Baseline models

**PT:** Todos os modelos usam a mesma divisão para que a comparação seja justa. KNN e regressão logística recebem escalonamento dentro de um `Pipeline`, sem vazamento da validação.

**EN:** All models use the same split for a fair comparison. KNN and logistic regression receive scaling inside a `Pipeline`, with no validation leakage.

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'KNN': Pipeline([('scaler', StandardScaler()), ('model', KNeighborsClassifier(n_neighbors=5))]),
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    results.append({'model': name, 'validation_accuracy': accuracy_score(y_valid, model.predict(X_valid))})

pd.DataFrame(results).sort_values('validation_accuracy', ascending=False)

## Ajuste do modelo final / Final-model tuning

**PT:** O grid é pequeno e reproduzível, pensado para aprendizado e execução razoável. A seleção usa acurácia média de validação cruzada, não a base de teste do Kaggle.

**EN:** The grid is small and reproducible, designed for learning and reasonable runtime. Selection uses mean cross-validation accuracy, not the Kaggle test set.

In [ ]:
search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid={
        'n_estimators': [200, 500],
        'max_depth': [4, 8, None],
        'min_samples_leaf': [1, 3],
        'max_features': ['sqrt', None],
    },
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
)
search.fit(X_train, y_train)

best_model = search.best_estimator_
validation_predictions = best_model.predict(X_valid)
print('Melhores parâmetros / Best parameters:', search.best_params_)
print(f'CV accuracy: {search.best_score_:.3f}')
print(f'Validation accuracy: {accuracy_score(y_valid, validation_predictions):.3f}')

In [ ]:
ConfusionMatrixDisplay(confusion_matrix(y_valid, validation_predictions)).plot()

## Treino final e persistência / Final training and persistence

**PT:** Após escolher os hiperparâmetros, o modelo é treinado novamente com todos os registros rotulados. O notebook 04 apenas o carrega para gerar a submissão.

**EN:** After choosing hyperparameters, the model is trained again on all labeled records. Notebook 04 only loads it to generate the submission.

In [ ]:
final_model = RandomForestClassifier(random_state=RANDOM_STATE, **search.best_params_)
final_model.fit(X, y)
joblib.dump({'model': final_model, 'features': X.columns.tolist()}, ARTIFACT_DIR / 'titanic_model.joblib')

print('Modelo salvo / Model saved:', (ARTIFACT_DIR / 'titanic_model.joblib').resolve())